# Natural Language Toolkit - Vader

In [8]:
from pymongo import MongoClient
import pandas as pd
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

In [9]:
# 1. Initialization Vader
analyzer = SentimentIntensityAnalyzer()

def analyze_sentiment_vader(text):
    scores = analyzer.polarity_scores(text)
    compound = scores["compound"]
    if compound >= 0.05:
        return "pos"
    elif compound <= -0.05:
        return "neg"
    else:
        return "neu"

In [10]:
# 2. DB Connection
client = MongoClient("mongodb://localhost:27017/")
db = client["sentiment_demo_vader"]
posts_collection = db["posts"]

In [11]:
# clean collection
posts_collection.delete_many({})

DeleteResult({'n': 5, 'ok': 1.0}, acknowledged=True)

In [ ]:
# 3. post
posts = [
    {"_id": 1, "text": "I really love this new project, it's amazing!"},
    {"_id": 2, "text": "This update is terrible and breaks everything."},
    {"_id": 3, "text": "The movie was just okay, notihing special."},
    {"_id": 4, "text": "Excellent service, I will definitely come back."},
    {"_id": 5, "text": "Worst experience ever, totally disappointed"}
]

posts_collection.insert_many(posts)

InsertManyResult([1, 2, 3, 4, 5], acknowledged=True)

In [13]:
# 4. Sentiment analyses
for post in posts_collection.find():
    sentiment = analyze_sentiment_vader(post["text"])
    posts_collection.update_one(
        {"_id": post["_id"]},
        {"$set": {"sentiment_vander": sentiment}}
    )

    print(f"Post ID {post['_id']} classified: {sentiment}")

Post ID 1 classified: pos
Post ID 2 classified: neg
Post ID 3 classified: pos
Post ID 4 classified: pos
Post ID 5 classified: neg


In [14]:
# 5. MongoDB Query
print("Post counting")
pipeline = [
    {"$group": {"_id": "$sentiment_vander", "count": {"$sum": 1}}}
]

for result in posts_collection.aggregate(pipeline):
    print(result)

Post counting
{'_id': 'neg', 'count': 2}
{'_id': 'pos', 'count': 3}


In [15]:
# 6. Dataframe
df = pd.DataFrame(list(posts_collection.find({}, {"_id": 1, "text":1, "sentiment_vander":1})))
print("Post dataframe:")
print(df)

Post dataframe:
   _id                                             text sentiment_vander
0    1    I really love this new project, it's amazing!              pos
1    2   This update is terrible and breaks everything.              neg
2    3       The movie was just oaky, notihing special.              pos
3    4  Excellent service, I will definitely come back.              pos
4    5      Worst experience ever, totally disappointed              neg
